<a href="https://colab.research.google.com/github/zzzzzssyy/ECON3916-33674-Statistical-Machine-Learning/blob/main/Lab%2012/%5BLab_12_%5D_OLS%2C_Hedonic_Pricing%2C_and_RMSE_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.tools.eval_measures import rmse
import matplotlib.pyplot as plt

# Step 1: Ingestion from external source
url = 'https://raw.githubusercontent.com/zzzzzssyy/ECON3916-33674-Statistical-Machine-Learning/main/Data/Zillow_ZHVI_2026_Micro.csv'
df = pd.read_csv(url)

In [9]:
df.head()

,Home_Value,Square_Footage,Property_Age,Distance_to_Transit,School_District_Rating
0,329705.74,1941.0,5.5,6.45,Excellent
1,183343.63,1364.3,35.2,2.15,Average
2,354551.73,2386.9,52.4,0.75,Good
3,325773.17,2192.1,50.2,5.25,Excellent
4,359743.12,3069.8,66.5,12.69,Excellent


In [10]:
# Step 2: Defining the formula
# Utilizing the R-style patsy formula interface allows for elegant, readable model specification
formula ='Home_Value ~ Square_Footage + Property_Age + Distance_to_Transit + School_District_Rating'

In [11]:
# Step 3: Fitting the model and printing the summary
model = smf.ols(formula=formula,data=df)
results = model.fit()
print(results.summary())

                            OLS Regression Results                            
Dep. Variable:             Home_Value   R-squared:                       0.766
Model:                            OLS   Adj. R-squared:                  0.765
Method:                 Least Squares   F-statistic:                     542.5
Date:                Mon, 16 Mar 2026   Prob (F-statistic):          2.81e-309
Time:                        19:57:45   Log-Likelihood:                -12072.
No. Observations:                1000   AIC:                         2.416e+04
Df Residuals:                     993   BIC:                         2.419e+04
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                                          coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------
In

In [12]:
# Step 4: Generating predictions
# We extract the predicted values vector to transition from explanation to prediction
y_pred = results.predict(df)

In [14]:
# Step 5: Calculate RMSE between the actuals and the predictions
model_rmse = rmse(df["Home_Value"],y_pred)
print(f"\nThe Predictive RMSE is: ${model_rmse:,.2f}")


The Predictive RMSE is: $42,316.69


 AI-Assisted Expansion

In [18]:
import plotly.express as px

# ── Step 1: Extract residuals and fitted values from the statsmodels results object ──
# results.resid  → a pandas Series of (y_actual - y_fitted) for every observation
# results.fittedvalues → the ŷ vector OLS computed during model.fit()
residuals   = results.resid          # e_i = y_i - ŷ_i
fitted_vals = results.fittedvalues   # ŷ_i

# ── Step 2: Identify outliers (|residual| > 2 standard deviations) ──
resid_std   = residuals.std()        # σ̂ of the residual distribution
resid_mean  = residuals.mean()       # should be ≈ 0 for OLS

# Boolean mask: True when the observation exceeds the 2σ threshold
is_outlier  = (np.abs(residuals - resid_mean) > 2 * resid_std)

# ── Step 3: Build a plotting DataFrame with an outlier label column ──
plot_df = pd.DataFrame({
    "Fitted Values":  fitted_vals,
    "Residuals":      residuals,
    "Outlier":        np.where(is_outlier, "Outlier (>2σ)", "Normal")
})

# ── Step 4: Assign colors — normal points in steel blue, outliers in crimson ──
color_map = {
    "Normal":        "steelblue",
    "Outlier (>2σ)": "crimson"
}

# ── Step 5: Build the scatter plot with Plotly Express ──
fig = px.scatter(
    plot_df,
    x="Fitted Values",
    y="Residuals",
    color="Outlier",
    color_discrete_map=color_map,
    title="<b>Residual Forensics Dashboard</b> — Fitted Values vs. Residuals",
    labels={
        "Fitted Values": "Fitted (Predicted) Home Values  ŷ  ($)",
        "Residuals":     "Residual Error  e = y − ŷ  ($)"
    },
    hover_data={"Fitted Values": ":,.0f", "Residuals": ":,.0f"},
    opacity=0.65,
    template="plotly_white"
)

# ── Step 6: Add the horizontal zero-line (the "ideal" reference) ──
# If OLS assumptions hold, residuals should scatter randomly around this line
fig.add_hline(
    y=0,
    line_dash="dash",
    line_color="black",
    line_width=1.5,
    annotation_text="  Zero Error Line",
    annotation_position="bottom right"
)

# ── Step 7: Add ±2σ reference bands so the threshold is visually obvious ──
fig.add_hline(y= 2 * resid_std, line_dash="dot", line_color="crimson",
              line_width=1, annotation_text="  +2σ", annotation_position="top right")
fig.add_hline(y=-2 * resid_std, line_dash="dot", line_color="crimson",
              line_width=1, annotation_text="  −2σ", annotation_position="bottom right")

# ── Step 8: Polish layout ──
fig.update_layout(
    legend_title_text="Point Type",
    font=dict(size=13),
    title_font_size=16,
    width=900, height=550
)

fig.show()

# ── Step 9: Print a quick outlier summary ──
n_outliers = is_outlier.sum()
print(f"\n📊 Residual Summary")
print(f"   Total observations : {len(residuals):,}")
print(f"   Residual std dev   : ${resid_std:,.2f}")
print(f"   Outliers (>2σ)     : {n_outliers}  ({100*n_outliers/len(residuals):.1f}%)")


📊 Residual Summary
   Total observations : 1,000
   Residual std dev   : $42,337.87
   Outliers (>2σ)     : 37  (3.7%)
